# extract and preprocess the AF of SNPs from rc file

1. run for each chromosome
2. then merge all chromosomes data to get the entire genome-wide data


In [124]:

import re
import numpy as np
import pandas as pd


In [ ]:
# ===== Input / Output =====
#chr1
# rc_path = "../../../../../../analysis/variant_explore/illumina_HC_wgs_XJ4126/06-FST_removed_s16_s17/06-09_poola_AF_perSNP/hcontortus_chr1_Celeg_TT_arrow_pilon_rm_s16_s17-AF_rc"
# out_tsv = "../../../../../../analysis/variant_explore/illumina_HC_wgs_XJ4126/06-FST_removed_s16_s17/06-09_poola_AF_perSNP/01_AF_PCA/AF_major_chr1.tsv"

#chr2
# rc_path = "../../../../../../analysis/variant_explore/illumina_HC_wgs_XJ4126/06-FST_removed_s16_s17/06-09_poola_AF_perSNP/hcontortus_chr2_Celeg_TT_arrow_pilon_rm_s16_s17-AF_rc"
# out_tsv = "../../../../../../analysis/variant_explore/illumina_HC_wgs_XJ4126/06-FST_removed_s16_s17/06-09_poola_AF_perSNP/01_AF_PCA/AF_major_chr2.tsv"

# #chr3
# rc_path = "../../../../../../analysis/variant_explore/illumina_HC_wgs_XJ4126/06-FST_removed_s16_s17/06-09_poola_AF_perSNP/hcontortus_chr3_Celeg_TT_arrow_pilon_rm_s16_s17-AF_rc"
# out_tsv = "../../../../../../analysis/variant_explore/illumina_HC_wgs_XJ4126/06-FST_removed_s16_s17/06-09_poola_AF_perSNP/01_AF_PCA/AF_major_chr3.tsv"

# #chr4
# rc_path = "../../../../../../analysis/variant_explore/illumina_HC_wgs_XJ4126/06-FST_removed_s16_s17/06-09_poola_AF_perSNP/hcontortus_chr4_Celeg_TT_arrow_pilon_rm_s16_s17-AF_rc"
# out_tsv = "../../../../../../analysis/variant_explore/illumina_HC_wgs_XJ4126/06-FST_removed_s16_s17/06-09_poola_AF_perSNP/01_AF_PCA/AF_major_chr4.tsv"

# #chr5
# rc_path = "../../../../../../analysis/variant_explore/illumina_HC_wgs_XJ4126/06-FST_removed_s16_s17/06-09_poola_AF_perSNP/hcontortus_chr5_Celeg_TT_arrow_pilon_rm_s16_s17-AF_rc"
# out_tsv = "../../../../../../analysis/variant_explore/illumina_HC_wgs_XJ4126/06-FST_removed_s16_s17/06-09_poola_AF_perSNP/01_AF_PCA/AF_major_chr5.tsv"

# #chrX
rc_path = "../../../../../../analysis/variant_explore/illumina_HC_wgs_XJ4126/06-FST_removed_s16_s17/06-09_poola_AF_perSNP/hcontortus_chrX_Celeg_TT_arrow_pilon_rm_s16_s17-AF_rc"
out_tsv = "../../../../../../analysis/variant_explore/illumina_HC_wgs_XJ4126/06-FST_removed_s16_s17/06-09_poola_AF_perSNP/01_AF_PCA/AF_major_chrX.tsv"


# ===== Filters =====
#min_cov = 15              # per pool minimum coverage, I do not think we need to filter that again since we have already filtered by min coverage when we run the previous steps
any_pool_minor_cutoff = 0.05         # keep SNP if MAF >= 0.05
max_missing_frac = 0.2    # allow <=20% pools missing per SNP
keep_only_biallelic = True  # keep only biallelic SNPs
keep_only_del0 = True       # use true based on the man pages: if not, the position may not be reliable


In [126]:
df = pd.read_csv(rc_path, sep="\t", header=0)



In [127]:
df

,##chr,pos,rc,allele_count,allele_states,deletion_sum,snp_type,major_alleles(maa),minor_alleles(mia),maa_1,...,mia_6,mia_7,mia_8,mia_9,mia_10,mia_11,mia_12,mia_13,mia_14,mia_15
0,hcontortus_chrX_Celeg_TT_arrow_pilon,2535,A,2,G/A,0,rc,NNAAAAGNGGNNAGN,NNGGGGANAANNGAN,0/14,...,10/24,3/19,0/10,7/23,7/20,0/6,0/9,11/23,5/22,0/13
1,hcontortus_chrX_Celeg_TT_arrow_pilon,2581,T,2,T/C,0,rc,NNTCTTTNTNNNTTT,NNNTNNNNNNNNNNN,0/11,...,0/27,0/18,0/13,0/19,0/13,0/6,0/7,0/18,0/24,0/15
2,hcontortus_chrX_Celeg_TT_arrow_pilon,2703,C,2,C/T,0,rc,CCCTCCCNCCNNCCC,NNNCNNNNNNNNNTN,26/26,...,0/22,0/25,0/12,0/26,0/25,0/9,0/13,0/25,5/34,0/19
3,hcontortus_chrX_Celeg_TT_arrow_pilon,2867,T,2,T/C,0,rc,CNTTTTCCCCNNCTC,TNCCCCTTTTNNTCT,13/20,...,13/29,7/22,8/18,6/22,5/19,0/8,0/7,11/25,13/28,7/17
4,hcontortus_chrX_Celeg_TT_arrow_pilon,3307,T,2,C/T,0,rc,CTCCTCCCCTNTTCN,TCTNCTTTTCNCCTN,16/31,...,16/37,5/31,6/22,1/20,8/23,0/11,5/16,9/26,14/48,0/12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1613629,hcontortus_chrX_Celeg_TT_arrow_pilon,46012368,G,2,A/G,0,rc,NNNNANANANNNNNN,NNNNGNGNGNNNNNN,0/4,...,0/8,5/24,0/11,8/16,0/3,0/7,0/5,0/12,0/9,0/14
1613630,hcontortus_chrX_Celeg_TT_arrow_pilon,46012377,T,2,A/T,0,rc,NAAAANANANNNAAA,NTTNTNTNTNNNTTT,0/12,...,0/14,2/32,0/14,1/24,0/7,0/11,0/11,5/15,2/18,4/24
1613631,hcontortus_chrX_Celeg_TT_arrow_pilon,46012381,G,2,G/C,0,rc,GGGCGCGGGNNNGGC,CCCGCGCCCNNNCCG,9/16,...,6/16,13/37,6/21,7/29,0/12,0/12,0/9,5/19,11/24,11/22
1613632,hcontortus_chrX_Celeg_TT_arrow_pilon,46012384,T,2,T/C,0,rc,TTTTTTCCCNNNTTT,CCCCCCTTTNNNCCC,11/16,...,7/17,16/39,9/20,11/29,0/12,0/12,0/12,8/18,9/23,9/26


In [128]:

df.columns.tolist()

['##chr',
 'pos',
 'rc',
 'allele_count',
 'allele_states',
 'deletion_sum',
 'snp_type',
 'major_alleles(maa)',
 'minor_alleles(mia)',
 'maa_1',
 'maa_2',
 'maa_3',
 'maa_4',
 'maa_5',
 'maa_6',
 'maa_7',
 'maa_8',
 'maa_9',
 'maa_10',
 'maa_11',
 'maa_12',
 'maa_13',
 'maa_14',
 'maa_15',
 'mia_1',
 'mia_2',
 'mia_3',
 'mia_4',
 'mia_5',
 'mia_6',
 'mia_7',
 'mia_8',
 'mia_9',
 'mia_10',
 'mia_11',
 'mia_12',
 'mia_13',
 'mia_14',
 'mia_15']

In [129]:
print("Raw shape:", df.shape)

Raw shape: (1613634, 39)


## filter:  `keep_only_biallelic SNPs` and `keep_only_deletion sum = 0`



In [130]:
df_f = df.copy()

if keep_only_biallelic and "allele_count" in df_f.columns:
    df_f = df_f[df_f["allele_count"].astype(int) == 2]

if keep_only_del0 and "deletion_sum" in df_f.columns:
    df_f = df_f[df_f["deletion_sum"].astype(float) == 0]

print("After del0 + biallelic:", df_f.shape)
df_f[["allele_count", "allele_states", "snp_type"]].head()



After del0 + biallelic: (1523969, 39)


,allele_count,allele_states,snp_type
0,2,G/A,rc
1,2,T/C,rc
2,2,C/T,rc
3,2,T/C,rc
4,2,C/T,rc


In [131]:
df_f.iloc[:, 9:24].head()  

,maa_1,maa_2,maa_3,maa_4,maa_5,maa_6,maa_7,maa_8,maa_9,maa_10,maa_11,maa_12,maa_13,maa_14,maa_15
0,0/14,0/9,15/25,10/15,13/18,14/24,16/19,0/10,16/23,13/20,0/6,0/9,12/23,17/22,0/13
1,0/11,0/9,15/15,16/18,20/20,27/27,18/18,0/13,19/19,0/13,0/6,0/7,18/18,24/24,15/15
2,26/26,26/26,28/28,14/23,25/25,22/22,25/25,0/12,26/26,25/25,0/9,0/13,25/25,29/34,19/19
3,13/20,0/13,24/33,24/29,19/22,16/29,15/22,10/18,16/22,14/19,0/8,0/7,14/25,15/28,10/17
4,16/31,23/39,15/27,28/28,18/22,21/37,26/31,16/22,19/20,15/23,0/11,11/16,17/26,34/48,0/12


In [132]:
df_f.iloc[:, 24:].head()  

,mia_1,mia_2,mia_3,mia_4,mia_5,mia_6,mia_7,mia_8,mia_9,mia_10,mia_11,mia_12,mia_13,mia_14,mia_15
0,0/14,0/9,10/25,5/15,5/18,10/24,3/19,0/10,7/23,7/20,0/6,0/9,11/23,5/22,0/13
1,0/11,0/9,0/15,2/18,0/20,0/27,0/18,0/13,0/19,0/13,0/6,0/7,0/18,0/24,0/15
2,0/26,0/26,0/28,9/23,0/25,0/22,0/25,0/12,0/26,0/25,0/9,0/13,0/25,5/34,0/19
3,7/20,0/13,9/33,5/29,3/22,13/29,7/22,8/18,6/22,5/19,0/8,0/7,11/25,13/28,7/17
4,15/31,16/39,12/27,0/28,4/22,16/37,5/31,6/22,1/20,8/23,0/11,5/16,9/26,14/48,0/12


## define the unified allele: use the first allele in allele_states (global major)

In [133]:
global_major = (
    df_f["allele_states"]
    .astype(str)
    .str.split("/", expand=True)[0]
    .to_numpy()
) # We convert it to numpy array for easier manipulation later.

global_major[:10]


array(['G', 'T', 'C', 'T', 'C', 'G', 'A', 'A', 'C', 'G'], dtype=object)

## find maa_i , mia_i cols

In [134]:

maa_cols = [c for c in df_f.columns if re.fullmatch(r"maa_\d+", str(c))] # str(c) to avoid warning about non-string column names
mia_cols = [c for c in df_f.columns if re.fullmatch(r"mia_\d+", str(c))]

npools = len(maa_cols)  # number of pools
print("npools =", npools)
print("maa example:", maa_cols) ## based on bamlist_removed_s16_s17.txt, 1-15 pools = 1-15 sample id
print("mia example:", mia_cols)


npools = 15
maa example: ['maa_1', 'maa_2', 'maa_3', 'maa_4', 'maa_5', 'maa_6', 'maa_7', 'maa_8', 'maa_9', 'maa_10', 'maa_11', 'maa_12', 'maa_13', 'maa_14', 'maa_15']
mia example: ['mia_1', 'mia_2', 'mia_3', 'mia_4', 'mia_5', 'mia_6', 'mia_7', 'mia_8', 'mia_9', 'mia_10', 'mia_11', 'mia_12', 'mia_13', 'mia_14', 'mia_15']


## turn the maa_i and mia_i columns （ "count/coverage"） into numeric matrices.


In [135]:

""" create 3 matrices:

maj_count[i, j]: major allele count of SNP i in pool j

min_count[i, j]: minor allele count of SNP i in pool j

cov[i, j]: coverage of SNP i in pool j (taken from the denominator of maa_i) 
"""


def split_count_cov(series): # input is a pandas Series of strings like "count/cov", will be the entire column of maa_i or mia_i
    s = series.astype(str).str.strip()
    parts = s.str.split("/", n=1, expand=True)
    count = pd.to_numeric(parts[0], errors="coerce")
    cov   = pd.to_numeric(parts[1], errors="coerce")
    return count.to_numpy(dtype=float), cov.to_numpy(dtype=float)

n_snp = df_f.shape[0] # number of SNPs after filtering (rows)

maj_count = np.zeros((n_snp, npools), dtype=float)  # npools is number of pools
min_count = np.zeros((n_snp, npools), dtype=float)
cov = np.zeros((n_snp, npools), dtype=float)


for j, c in enumerate(maa_cols):
    cnt, cv = split_count_cov(df_f[c])
    maj_count[:, j] = cnt
    cov[:, j] = cv

for j, c in enumerate(mia_cols):
    cnt, _ = split_count_cov(df_f[c])
    min_count[:, j] = cnt

print("coverage min/max:", np.nanmin(cov), np.nanmax(cov))


coverage min/max: 0.0 497.0


In [136]:
maj_count[:8, :15]

array([[ 0.,  0., 15., 10., 13., 14., 16.,  0., 16., 13.,  0.,  0., 12.,
        17.,  0.],
       [ 0.,  0., 15., 16., 20., 27., 18.,  0., 19.,  0.,  0.,  0., 18.,
        24., 15.],
       [26., 26., 28., 14., 25., 22., 25.,  0., 26., 25.,  0.,  0., 25.,
        29., 19.],
       [13.,  0., 24., 24., 19., 16., 15., 10., 16., 14.,  0.,  0., 14.,
        15., 10.],
       [16., 23., 15., 28., 18., 21., 26., 16., 19., 15.,  0., 11., 17.,
        34.,  0.],
       [33., 45., 25., 40., 28., 41., 36., 27., 26., 28., 16., 18., 33.,
        52., 16.],
       [31., 45., 26., 25., 36., 61., 32., 30., 34., 42., 17., 17., 36.,
        50., 18.],
       [30., 40., 20., 32., 32., 57., 38., 32., 36., 40., 21., 18., 36.,
        49., 15.]])

In [137]:
min_count

array([[ 0.,  0., 10., ..., 11.,  5.,  0.],
       [ 0.,  0.,  0., ...,  0.,  0.,  0.],
       [ 0.,  0.,  0., ...,  0.,  5.,  0.],
       ...,
       [ 7.,  3.,  7., ...,  5., 11., 11.],
       [ 5.,  7.,  8., ...,  8.,  9.,  9.],
       [ 6.,  6.,  6., ...,  4., 10., 10.]], shape=(1523969, 15))

In [138]:
cov[:10, :15]

array([[14.,  9., 25., 15., 18., 24., 19., 10., 23., 20.,  6.,  9., 23.,
        22., 13.],
       [11.,  9., 15., 18., 20., 27., 18., 13., 19., 13.,  6.,  7., 18.,
        24., 15.],
       [26., 26., 28., 23., 25., 22., 25., 12., 26., 25.,  9., 13., 25.,
        34., 19.],
       [20., 13., 33., 29., 22., 29., 22., 18., 22., 19.,  8.,  7., 25.,
        28., 17.],
       [31., 39., 27., 28., 22., 37., 31., 22., 20., 23., 11., 16., 26.,
        48., 12.],
       [34., 45., 25., 43., 28., 45., 36., 27., 26., 28., 16., 18., 33.,
        52., 16.],
       [31., 45., 26., 37., 36., 61., 32., 30., 34., 42., 17., 17., 36.,
        50., 18.],
       [30., 42., 20., 32., 32., 57., 38., 32., 36., 41., 21., 18., 36.,
        53., 15.],
       [33., 33., 21., 27., 32., 53., 34., 32., 29., 37., 22., 22., 33.,
        49., 14.],
       [42., 48., 23., 57., 37., 63., 32., 34., 43., 24., 16., 30., 32.,
        51., 20.]])

In [139]:
# check the coverage values, but see there are some coverage <15 (our min cov in the code); but each SNP will not have all pools with low coverage based on the code, so we should keep them

print("cov overall min/max:", np.nanmin(cov), np.nanmax(cov))  

cov overall min/max: 0.0 497.0


## calculate global major allele's AF in each pool

In [140]:
# get the major alleles and minor alleles as strings
maa_str = df_f["major_alleles(maa)"].astype(str).to_numpy()
mia_str = df_f["minor_alleles(mia)"].astype(str).to_numpy()

In [141]:
maa_str

array(['NNAAAAGNGGNNAGN', 'NNTCTTTNTNNNTTT', 'CCCTCCCNCCNNCCC', ...,
       'GGGCGCGGGNNNGGC', 'TTTTTTCCCNNNTTT', 'GGGAGGGGGNNNGGG'],
      shape=(1523969,), dtype=object)

In [142]:
## create a matrix with Nan (same shape as maj_count)
global_major_count = np.full_like(maj_count, np.nan, dtype=float)
global_major_count

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]], shape=(1523969, 15))

In [143]:
### calculate the count of the global major allele in each pool for each SNP based on the rules below:

""" 
the rules to determine the count of the global major allele in each pool for each SNP: major(m), minor(n), global major (gm)

    if m==N and n==N → NA

    if global_major==major allele → use major count

    if global_major==minor allele → use minor count

    if gm != major(m) and gm != minor(n):

        if m == "N" or n == "N", means one of the allele is missing, we cannot determine if the global major allele is present or not, → NA

        if m and n are both not "N"  → 0

"""




for i in range(df_f.shape[0]):   # loop through each SNP (row number)
    gm = global_major[i]
    ma = maa_str[i]
    mi = mia_str[i]

    for j in range(npools): # loop through each pool (pop column number)
        m = ma[j]
        n = mi[j]
        if m == "N" and n == "N": # skip and go to next loop iteration (e.g. includes when coverage is 0), to keep NAN in matrix
            continue

        # check if the global major allele is the same as the major allele or minor allele in this pool, and assign the count accordingly
        if gm == m:
            global_major_count[i, j] = maj_count[i, j]
        elif gm == n:
            global_major_count[i, j] = min_count[i, j]
        else: # when gm != m and gm != n:
            if m == "N" or n == "N":
                continue
            global_major_count[i, j] = 0.0


In [144]:
## calculate the allele frequency of the global major allele in each pool

global_major_af = global_major_count / cov  # allele frequency of the global major allele in each pool for each SNP
global_major_af[~np.isfinite(global_major_af)] = np.nan # transfer inf and -inf to nan. just in case

print("AF shape:", global_major_af.shape)
print("AF min/max:", np.nanmin(global_major_af), np.nanmax(global_major_af))

AF shape: (1523969, 15)
AF min/max: 0.0 1.0


In [145]:
global_major_count

array([[nan, nan, 10., ..., 11., 17., nan],
       [nan, nan, 15., ..., 18., 24., 15.],
       [26., 26., 28., ..., 25., 29., 19.],
       ...,
       [ 9., 17., 13., ..., 14., 13., 11.],
       [11., 18., 11., ..., 10., 14., 17.],
       [13., 19., 13., ..., 16., 18., 15.]], shape=(1523969, 15))

In [146]:
global_major_af

array([[       nan,        nan, 0.4       , ..., 0.47826087, 0.77272727,
               nan],
       [       nan,        nan, 1.        , ..., 1.        , 1.        ,
        1.        ],
       [1.        , 1.        , 1.        , ..., 1.        , 0.85294118,
        1.        ],
       ...,
       [0.5625    , 0.85      , 0.65      , ..., 0.73684211, 0.54166667,
        0.5       ],
       [0.6875    , 0.72      , 0.57894737, ..., 0.55555556, 0.60869565,
        0.65384615],
       [0.68421053, 0.76      , 0.68421053, ..., 0.8       , 0.64285714,
        0.6       ]], shape=(1523969, 15))

### check the global_major_count and global_major_af : nan and zero

In [147]:
M = global_major_count
M = global_major_af

n_total = M.size

print("Total entries:", n_total)
print("NaN:", np.isnan(M).sum())
print("0:", (M == 0).sum())


Total entries: 22859535
NaN: 768879
0: 163


In [148]:
# how many percentage of Nan
(3542059/50426685)*100

7.024175791051901

In [149]:

M = global_major_af

n_total = M.size

print("Total entries:", n_total)
print("NaN:", np.isnan(M).sum())
print("0:", (M == 0).sum())

Total entries: 22859535
NaN: 768879
0: 163


In [150]:
## check how many times a pool doesnot have the global major allele as either major or minor allele, 
# # and how many times a pool has no allele call at a SNP (e.g. coverage is 0, or both major and minor allele are N)

other_allele = 0
nothing = 0

for i in range(df_f.shape[0]):   # loop through each SNP (row number)
    gm = global_major[i]
    ma = maa_str[i]
    mi = mia_str[i]

    for j in range(npools): # loop through each pool (pop column number)
        m = ma[j]
        n = mi[j]
        if m == "N" and n == "N":
            nothing += 1
            continue


        if gm != m and gm != n:
            other_allele += 1
        

print("how many times a pool has no allele call at a SNP:" ,nothing)
print("how many times does a pool not have the global major allele as either major or minor allele (this pool has other alleles):" ,other_allele)

how many times a pool has no allele call at a SNP: 741532
how many times does a pool not have the global major allele as either major or minor allele (this pool has other alleles): 27510


In [151]:
## calculate the percentage of pools doesnot have the global major allele as either major or minor allele: 1.6 %
(56112/3361779)*100

1.6691162625502747

## filter out missing value NAN

In [152]:
## calculate how many percentage of SNPs have 0-15 missing pools
### output: around 73/78% SNPs have no missing value!! 
# so I would filter out all SNPs with any missing value, and keep only the 73% SNPs with complete data for all pools for PCA

k_missing = np.sum(np.isnan(global_major_af), axis=1) # array: number of NaN (missing) values across pools for each SNP

tab_k = pd.Series(k_missing).value_counts(normalize=True).sort_index() # calculate percentage of SNPs with k missing pools, 0.7=70%

tab_k.index = [f"{k}/{npools} pools missing" for k in tab_k.index] # change index to be more informative, e.g. "0/15 pools missing", "1/15 pools missing" etc.
print(tab_k)


0/15 pools missing     0.817703
1/15 pools missing     0.093007
2/15 pools missing     0.033096
3/15 pools missing     0.015360
4/15 pools missing     0.009195
5/15 pools missing     0.006289
6/15 pools missing     0.004780
7/15 pools missing     0.003910
8/15 pools missing     0.003268
9/15 pools missing     0.002955
10/15 pools missing    0.002640
11/15 pools missing    0.002415
12/15 pools missing    0.002079
13/15 pools missing    0.001919
14/15 pools missing    0.001383
Name: proportion, dtype: float64


In [153]:
## filter out SNPs with any missing value

keep_no_missing = ~np.any(np.isnan(global_major_af), axis=1) # True for SNPs with no missing value, False for SNPs with any missing value
keep_no_missing[:10]

array([False, False, False, False, False,  True,  True,  True, False,
        True])

In [154]:
df_f_no_missing = df_f.loc[keep_no_missing].copy()
global_major_af_no_missing = global_major_af[keep_no_missing, :]


print("After no-missing filter:", df_f_no_missing.shape)


After no-missing filter: (1246154, 39)


In [155]:
# recheck the percentage
(2478565/3361779)*100

73.72777925021246

## keep SNP existed among any of 15 pools at a MAF>= 0.05  

In [156]:
## each pool's minorAF = min(p, 1-p), p is af of the global major allele
## as in each pool, the global major allele is either the major allele or the minor allele

## calculate the minor allele frequency in each pool in each SNP
minor_af_per_pool = np.minimum(global_major_af_no_missing, 1.0 - global_major_af_no_missing)
minor_af_per_pool

array([[0.02941176, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.04761905, 0.        , ..., 0.        , 0.0754717 ,
        0.        ],
       ...,
       [0.        , 0.02040816, 0.        , ..., 0.06666667, 0.        ,
        0.        ],
       [0.06451613, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.03333333, 0.02702703, 0.        , ..., 0.03703704, 0.        ,
        0.        ]], shape=(1246154, 15))

In [157]:
## in each SNP row, get the maximum minor allele frequency across all pools, to see if any SNP has MAF >= 0.05 in any pool

max_minor = np.nanmax(minor_af_per_pool, axis=1)
max_minor

array([0.08888889, 0.32432432, 0.0754717 , ..., 0.11627907, 0.06451613,
       0.05405405], shape=(1246154,))

In [158]:
## check how many SNPs have MAF >= 0.05, 0.01, 0.1 in at least one pool, to make sure we are not losing too many SNPs if we choose the cutoff of 0.05
## output: we will choose to keep SNPs with MAF >= 0.05 in at least one pool （keep around 95%...）

cutoffs = [0.01, 0.05, 0.10]

n_total = len(max_minor)
for c in cutoffs:
    n = np.sum(max_minor >= c)
    print(f"max per-pool MAF >= {c:.2f}: {n} SNPs ({n/n_total*100:.1f}%)")

print("max_minor min/max:", np.nanmin(max_minor), np.nanmax(max_minor))


max per-pool MAF >= 0.01: 1246152 SNPs (100.0%)
max per-pool MAF >= 0.05: 1155583 SNPs (92.7%)
max per-pool MAF >= 0.10: 838532 SNPs (67.3%)
max_minor min/max: 0.0 0.5


In [159]:
## keep SNPs with MAF >= 0.05 in at least one pool 
keep_any_pool = max_minor >= 0.05

df_f_cleaned = df_f_no_missing.loc[keep_any_pool].copy()
global_major_af_cleaned = global_major_af_no_missing[keep_any_pool, :]

print("After any-pool minorAF filter:", df_f_cleaned.shape)
print("Max minorAF kept range:", np.nanmin(max_minor[keep_any_pool]), np.nanmax(max_minor[keep_any_pool]))

After any-pool minorAF filter: (1155583, 39)
Max minorAF kept range: 0.05 0.5


In [160]:
df_f_cleaned

,##chr,pos,rc,allele_count,allele_states,deletion_sum,snp_type,major_alleles(maa),minor_alleles(mia),maa_1,...,mia_6,mia_7,mia_8,mia_9,mia_10,mia_11,mia_12,mia_13,mia_14,mia_15
5,hcontortus_chrX_Celeg_TT_arrow_pilon,3407,G,2,G/A,0,pop,GGGGGGGGGGGGGGG,ANNANANNNNNNNNN,33/34,...,4/45,0/36,0/27,0/26,0/28,0/16,0/18,0/33,0/52,0/16
6,hcontortus_chrX_Celeg_TT_arrow_pilon,3520,A,2,A/G,0,pop,AAAAAAAAAAAAAAA,NNNGNNNNNNNNNNN,31/31,...,0/61,0/32,0/30,0/34,0/42,0/17,0/17,0/36,0/50,0/18
7,hcontortus_chrX_Celeg_TT_arrow_pilon,3549,A,2,A/T,0,pop,AAAAAAAAAAAAAAA,NTNNNNNNNTNNNTN,30/30,...,0/57,0/38,0/32,0/36,1/41,0/21,0/18,0/36,4/53,0/15
9,hcontortus_chrX_Celeg_TT_arrow_pilon,3914,G,2,G/A,0,pop,GGGGGGGGGGGGGGG,NNNNNNANNNNNNAN,42/42,...,0/63,2/32,0/34,0/43,0/24,0/16,0/30,0/32,2/51,0/20
10,hcontortus_chrX_Celeg_TT_arrow_pilon,3926,G,2,G/T,0,rc|pop,GGGTGGGGGGGGGGG,NNNGNNNNNNNNNNN,47/47,...,0/58,0/36,0/35,0/47,0/24,0/17,0/33,0/35,0/49,0/20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1613620,hcontortus_chrX_Celeg_TT_arrow_pilon,46009855,A,2,A/G,0,pop,AAAAAAAAAAAAAAA,NNGGNGNGGNGGNNN,33/33,...,2/53,0/27,2/27,2/29,0/32,2/23,1/20,0/33,0/42,0/29
1613621,hcontortus_chrX_Celeg_TT_arrow_pilon,46009857,T,2,T/A,0,pop,TTTTTTTTTTTTTTT,NANNNNNNNANNNNN,36/36,...,0/52,0/27,0/27,0/29,3/31,0/23,0/20,0/34,0/40,0/28
1613624,hcontortus_chrX_Celeg_TT_arrow_pilon,46009875,G,2,G/A,0,pop,GGGGGGGGGGGGGGG,NTNANANNNNNNANN,35/35,...,1/50,0/26,0/23,0/30,0/24,0/21,0/18,1/30,0/47,0/28
1613625,hcontortus_chrX_Celeg_TT_arrow_pilon,46009883,T,2,T/C,0,pop,TTTTTTTTTTTTTTT,CNNNCNNNNNCNNNN,29/31,...,0/45,0/25,0/22,0/29,0/25,1/22,0/19,0/31,0/46,0/26


## save preprocessed AF data

In [161]:
meta_cols = ["##chr", "pos", "rc", "allele_states"]

out = df_f_cleaned[meta_cols].copy()
for j in range(global_major_af_cleaned.shape[1]):
    out[f"{j+1}"] = global_major_af_cleaned[:, j]

print(out.shape)
out.head()


(1155583, 19)


,##chr,pos,rc,allele_states,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
5,hcontortus_chrX_Celeg_TT_arrow_pilon,3407,G,G/A,0.970588,1.000000,1.0,0.930233,1.0,0.911111,1.0000,1.0,1.0,1.00000,1.0,1.0,1.0,1.000000,1.0
6,hcontortus_chrX_Celeg_TT_arrow_pilon,3520,A,A/G,1.000000,1.000000,1.0,0.675676,1.0,1.000000,1.0000,1.0,1.0,1.00000,1.0,1.0,1.0,1.000000,1.0
7,hcontortus_chrX_Celeg_TT_arrow_pilon,3549,A,A/T,1.000000,0.952381,1.0,1.000000,1.0,1.000000,1.0000,1.0,1.0,0.97561,1.0,1.0,1.0,0.924528,1.0
9,hcontortus_chrX_Celeg_TT_arrow_pilon,3914,G,G/A,1.000000,1.000000,1.0,1.000000,1.0,1.000000,0.9375,1.0,1.0,1.00000,1.0,1.0,1.0,0.960784,1.0
10,hcontortus_chrX_Celeg_TT_arrow_pilon,3926,G,G/T,1.000000,1.000000,1.0,0.327586,1.0,1.000000,1.0000,1.0,1.0,1.00000,1.0,1.0,1.0,1.000000,1.0


In [162]:
## rename sample ID

map = {
  "1":"A1",
  "6":"A2",
  "8":"B",
  "3":"C",
  "4":"D",
  "5":"E",
  "2":"F1",
  "10":"F2",
  "7":"G1",
  "9":"G2",
  "11":"H",
  "13":"I",
  "15":"K",
  "14":"L",
  "12":"N"
}

out=out.rename(columns=map)
out.head()

,##chr,pos,rc,allele_states,A1,F1,C,D,E,A2,G1,B,G2,F2,H,N,I,L,K
5,hcontortus_chrX_Celeg_TT_arrow_pilon,3407,G,G/A,0.970588,1.000000,1.0,0.930233,1.0,0.911111,1.0000,1.0,1.0,1.00000,1.0,1.0,1.0,1.000000,1.0
6,hcontortus_chrX_Celeg_TT_arrow_pilon,3520,A,A/G,1.000000,1.000000,1.0,0.675676,1.0,1.000000,1.0000,1.0,1.0,1.00000,1.0,1.0,1.0,1.000000,1.0
7,hcontortus_chrX_Celeg_TT_arrow_pilon,3549,A,A/T,1.000000,0.952381,1.0,1.000000,1.0,1.000000,1.0000,1.0,1.0,0.97561,1.0,1.0,1.0,0.924528,1.0
9,hcontortus_chrX_Celeg_TT_arrow_pilon,3914,G,G/A,1.000000,1.000000,1.0,1.000000,1.0,1.000000,0.9375,1.0,1.0,1.00000,1.0,1.0,1.0,0.960784,1.0
10,hcontortus_chrX_Celeg_TT_arrow_pilon,3926,G,G/T,1.000000,1.000000,1.0,0.327586,1.0,1.000000,1.0000,1.0,1.0,1.00000,1.0,1.0,1.0,1.000000,1.0


In [163]:
# keep only the first allele in the "allele_states" column, which is the global major allele
# just remember that: the "allele_states" column is in the format of "major/minor", but the major and minor is gobal major and global minor allele concider all pools with good coverage (higher than cov threshold you set), might not the major and minor allele in each pool

out["rc"]=out["allele_states"].astype(str).str.split("/").str[0] 
out= out.rename(columns={"rc": "global_major_allele"})
out= out.rename(columns={"##chr": "chr"}) # remove ## to make R easier to read the column name
out.head()

,chr,pos,global_major_allele,allele_states,A1,F1,C,D,E,A2,G1,B,G2,F2,H,N,I,L,K
5,hcontortus_chrX_Celeg_TT_arrow_pilon,3407,G,G/A,0.970588,1.000000,1.0,0.930233,1.0,0.911111,1.0000,1.0,1.0,1.00000,1.0,1.0,1.0,1.000000,1.0
6,hcontortus_chrX_Celeg_TT_arrow_pilon,3520,A,A/G,1.000000,1.000000,1.0,0.675676,1.0,1.000000,1.0000,1.0,1.0,1.00000,1.0,1.0,1.0,1.000000,1.0
7,hcontortus_chrX_Celeg_TT_arrow_pilon,3549,A,A/T,1.000000,0.952381,1.0,1.000000,1.0,1.000000,1.0000,1.0,1.0,0.97561,1.0,1.0,1.0,0.924528,1.0
9,hcontortus_chrX_Celeg_TT_arrow_pilon,3914,G,G/A,1.000000,1.000000,1.0,1.000000,1.0,1.000000,0.9375,1.0,1.0,1.00000,1.0,1.0,1.0,0.960784,1.0
10,hcontortus_chrX_Celeg_TT_arrow_pilon,3926,G,G/T,1.000000,1.000000,1.0,0.327586,1.0,1.000000,1.0000,1.0,1.0,1.00000,1.0,1.0,1.0,1.000000,1.0


In [164]:
out.to_csv(out_tsv, sep="\t", index=False, float_format="%.4f")
out_tsv

'../../../../../../analysis/variant_explore/illumina_HC_wgs_XJ4126/06-FST_removed_s16_s17/06-09_poola_AF_perSNP/01_AF_PCA/AF_major_chrX.tsv'

## merge all chr AF data to get the entire genome-wide AF data

In [4]:
import pandas as pd
import glob
import os

In [ ]:
merge_out_tsv = "../../../../../../analysis/variant_explore/illumina_HC_wgs_XJ4126/06-FST_removed_s16_s17/06-09_poola_AF_perSNP/01_AF_PCA/AF_major_genome.tsv"

indir_all = "../../../../../../analysis/variant_explore/illumina_HC_wgs_XJ4126/06-FST_removed_s16_s17/06-09_poola_AF_perSNP/01_AF_PCA/"   # change this
files = sorted(glob.glob(os.path.join(indir_all, "AF_major_chr*.tsv")))
files

['../../../../../../analysis/variant_explore/illumina_HC_wgs_XJ4126/06-FST_removed_s16_s17/06-09_poola_AF_perSNP/01_AF_PCA/AF_major_chr1.tsv',
 '../../../../../../analysis/variant_explore/illumina_HC_wgs_XJ4126/06-FST_removed_s16_s17/06-09_poola_AF_perSNP/01_AF_PCA/AF_major_chr2.tsv',
 '../../../../../../analysis/variant_explore/illumina_HC_wgs_XJ4126/06-FST_removed_s16_s17/06-09_poola_AF_perSNP/01_AF_PCA/AF_major_chr3.tsv',
 '../../../../../../analysis/variant_explore/illumina_HC_wgs_XJ4126/06-FST_removed_s16_s17/06-09_poola_AF_perSNP/01_AF_PCA/AF_major_chr4.tsv',
 '../../../../../../analysis/variant_explore/illumina_HC_wgs_XJ4126/06-FST_removed_s16_s17/06-09_poola_AF_perSNP/01_AF_PCA/AF_major_chr5.tsv',
 '../../../../../../analysis/variant_explore/illumina_HC_wgs_XJ4126/06-FST_removed_s16_s17/06-09_poola_AF_perSNP/01_AF_PCA/AF_major_chrX.tsv']

In [7]:
# Read and merge (row-bind or vertical concatenation)
dfs = []
for f in files:
    d = pd.read_csv(f, sep="\t")
    dfs.append(d)

merged = pd.concat(dfs, ignore_index=True)


In [8]:
merged

,chr,pos,global_major_allele,allele_states,A1,F1,C,D,E,A2,G1,B,G2,F2,H,N,I,L,K
0,hcontortus_chr1_Celeg_TT_arrow_pilon,3805,A,A/G,0.9792,0.9423,0.9412,1.0000,1.0000,1.0000,0.9429,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
1,hcontortus_chr1_Celeg_TT_arrow_pilon,4124,A,A/T,0.3947,0.6071,0.4783,0.4750,0.6000,0.6207,0.5000,0.6774,0.5526,0.5676,0.8333,0.7143,0.4500,0.6061,0.9231
2,hcontortus_chr1_Celeg_TT_arrow_pilon,4635,G,G/A,1.0000,1.0000,1.0000,0.8333,1.0000,0.8824,1.0000,1.0000,1.0000,0.9600,1.0000,1.0000,1.0000,1.0000,1.0000
3,hcontortus_chr1_Celeg_TT_arrow_pilon,4636,T,T/C,0.9474,1.0000,1.0000,0.8611,0.9545,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.8947,0.9565,0.9375,1.0000
4,hcontortus_chr1_Celeg_TT_arrow_pilon,4655,C,C/T,1.0000,0.9259,1.0000,1.0000,1.0000,0.8889,0.9630,1.0000,1.0000,0.9643,1.0000,1.0000,1.0000,1.0000,0.8947
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14444521,hcontortus_chrX_Celeg_TT_arrow_pilon,46009855,A,A/G,1.0000,1.0000,0.9706,0.9444,1.0000,0.9623,1.0000,0.9259,0.9310,1.0000,0.9130,0.9500,1.0000,1.0000,1.0000
14444522,hcontortus_chrX_Celeg_TT_arrow_pilon,46009857,T,T/A,1.0000,0.8913,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.9032,1.0000,1.0000,1.0000,1.0000,1.0000
14444523,hcontortus_chrX_Celeg_TT_arrow_pilon,46009875,G,G/A,1.0000,0.9796,1.0000,0.8837,1.0000,0.9800,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.9333,1.0000,1.0000
14444524,hcontortus_chrX_Celeg_TT_arrow_pilon,46009883,T,T/C,0.9355,1.0000,1.0000,1.0000,0.9600,1.0000,1.0000,1.0000,1.0000,1.0000,0.9545,1.0000,1.0000,1.0000,1.0000


In [11]:
## save

merged.to_csv(merge_out_tsv,sep="\t", index=False, float_format="%.4f")